# 04_05 Classification - GBTClassifier
Train and evaluate GBTClassifier.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train GBTClassifier va visual ket qua danh gia tren test set.
- Muc tieu ky thuat: Hien thi metric table va confusion matrix trong notebook.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
spark=(SparkSession.builder.appName('04_05_gbt_cls').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'classification'/'gbt_classifier'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'classification_train')).select('order_id','label','features').dropna()
val_df=spark.read.parquet(str(FEATURE_DIR/'classification_val')).select('order_id','label','features').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'classification_test')).select('order_id','label','features').dropna()
gbt=GBTClassifier(featuresCol='features',labelCol='label',maxDepth=8,maxIter=60,stepSize=0.1,seed=42)
m=gbt.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
val_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_val)
val_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_val)
test_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_test)
test_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_test)
metrics={'model_family':'classification','model_name':'GBTClassifier','val_f1':float(val_f1),'val_accuracy':float(val_acc),'f1':float(test_f1),'accuracy':float(test_acc),'test_f1':float(test_f1),'test_accuracy':float(test_acc),'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count()}
print(metrics)
display(pd.DataFrame([metrics]))
cm_pdf=pred_test.groupBy('label','prediction').count().toPandas()
if not cm_pdf.empty:
    cm_table=cm_pdf.pivot(index='label', columns='prediction', values='count').fillna(0).sort_index().sort_index(axis=1)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_table, annot=True, fmt='.0f', cmap='Blues')
    plt.title('Confusion Matrix - GBTClassifier (Test)')
    plt.xlabel('Prediction')
    plt.ylabel('Label')
    plt.tight_layout()
    plt.show()
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'classification_gbt_classifier.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 00:52:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/03 00:52:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/03 00:52:56 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/03 00:52:56 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1000.0 KiB
26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1006.1 KiB
26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1006.8 KiB


26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1007.3 KiB
26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1008.0 KiB


26/04/03 00:53:40 WARN DAGScheduler: Broadcasting large task binary with size 1008.9 KiB
26/04/03 00:53:41 WARN DAGScheduler: Broadcasting large task binary with size 1010.4 KiB


...

{'model_family': 'classification', 'model_name': 'GBTClassifier', 'val_f1': 0.8524408059096721, 'val_accuracy': 0.8674577634754626, 'f1': 0.8585487882528205, 'accuracy': 0.8728881737731295, 'test_f1': 0.8585487882528205, 'test_accuracy': 0.8728881737731295, 'train_rows': 69609, 'val_rows': 14916, 'test_rows': 14916}


,model_family,model_name,val_f1,val_accuracy,f1,accuracy,test_f1,test_accuracy,train_rows,val_rows,test_rows
0,classification,GBTClassifier,0.852441,0.867458,0.858549,0.872888,0.858549,0.872888,69609,14916,14916


26/04/03 00:53:51 WARN DAGScheduler: Broadcasting large task binary with size 1157.5 KiB


26/04/03 00:53:52 WARN DAGScheduler: Broadcasting large task binary with size 1145.6 KiB
/var/folders/07/jh7lmpq57r72h5jdjy3vwy740000gn/T/ipykernel_5458/3527680636.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


341